# 02 · Casting Product - Preprocessing & DataLoaders

Data preparation for Transfer Learning:
- Resize → 224×224 (ImageNet standard)
- Augmentation on train (flip, rotation)
- Normalisation with ImageNet statistics

In [ ]:
import sys
from pathlib import Path

# Add notebooks/utils to path (works on Mac, Windows, Linux)
_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)

print('arkon_utils loaded ✓')

In [ ]:
import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Detect GPU
device = get_device()

print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')


In [ ]:
DATA_DIR  = Path('../../../data/03_casting/raw')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = recommended_num_workers(device)  # 0 on CPU/notebooks, 4 on CUDA
PIN_MEMORY  = device.type == 'cuda'            # speeds up GPU transfer
SEED        = 42
ASSETS      = 'cv/casting'
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)


## 1. Transforms - augmentation and normalisation

ImageNet mean/std - the standard for all pretrained torchvision models.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Train transforms:', train_transforms)
print('\nVal/Test transforms:', val_transforms)

## 2. Datasets - ImageFolder

`ImageFolder` derives the classes automatically from the folder names.

In [ ]:
full_train = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
test_set   = datasets.ImageFolder(root=TEST_DIR,  transform=val_transforms)

# Train/Val split 80/20
val_size   = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(SEED))
# val uses val_transforms (no augmentation)
val_set.dataset.transform = val_transforms

CLASSES    = full_train.classes
NUM_CLASSES = len(CLASSES)
print(f'Classes: {CLASSES} → {full_train.class_to_idx}')
print(f'Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')

## 3. DataLoaders

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')
print(f'PIN_MEMORY   : {PIN_MEMORY}  (True = faster GPU transfer)')


## 4. Sanity Check - batch check

In [ ]:
images, labels = next(iter(train_loader))
print(f'Batch images shape: {images.shape}')  # [32, 3, 224, 224]
print(f'Batch labels shape: {labels.shape}')  # [32]
print(f'Label classes: {[CLASSES[l] for l in labels[:8].tolist()]}')
print(f'Pixel range: [{images.min():.2f}, {images.max():.2f}]')

In [ ]:
# Visualize denormalized batch
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = denormalize(images[i]).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(CLASSES[labels[i].item()], fontsize=8)
    ax.axis('off')
plt.suptitle('Train Batch Sample (denormalized)')
plt.tight_layout()
save_figure(fig, 'casting_preprocessing_batch', subfolder=ASSETS)
plt.show()

## Summary

| Parameter | Value |
|---|---|
| Input size | 224×224 |
| Batch size | 32 |
| Train aug | Flip + Rotation + ColorJitter |
| Normalize | ImageNet stats |

➡️ **Next step:** `03_casting_modeling.ipynb`